# MediFlow AI: High-Performance GPU Model Training on Kaggle

This notebook contains the pipeline for training, optimizing, and evaluating the **XGBoost Admission Classifier** and **Prophet Bed Occupancy Forecaster** on Kaggle with GPU acceleration and hyperparameter tuning (via **Optuna**) to achieve **>90% classification accuracy/performance**.

In [ ]:
# 1. Setup and Dependencies
!pip install -q optuna prophet xgboost joblib scikit-learn pandas numpy faker

import os
import math
import hashlib
import joblib
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from prophet import Prophet
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, confusion_matrix
from faker import Faker

print("Setup complete. XGBoost version:", xgb.__version__)
print("Optuna version:", optuna.__version__)

## 2. Data Loading & Replicating Feature Pipeline

Here we search for the raw CSV dataset and reconstruct the deterministic feature pipeline including synthetic variables (ICU capacity, doctor availability, ambulance requests) matching the production environment.

In [ ]:
def find_data_file():
    # Standard pathways for local and Kaggle environments
    paths = [
        "../datasets/Hospital ER_Data.csv",
        "datasets/Hospital ER_Data.csv",
        "/kaggle/input/hospital-er-data/Hospital ER_Data.csv",
        "/kaggle/input/mediflow-dataset/Hospital ER_Data.csv"
    ]
    for p in paths:
        if os.path.exists(p):
            print(f"Found dataset at: {p}")
            return p
    raise FileNotFoundError("Dataset CSV file not found. Please upload 'Hospital ER_Data.csv' to Kaggle.")

def get_deterministic_seed(timestamp_str: str) -> int:
    return int(hashlib.md5(timestamp_str.encode("utf-8")).hexdigest(), 16) % 100000000

def derive_severity_level(wait_time: int, department: str) -> int:
    dept = str(department).lower()
    if "card" in dept or "icu" in dept or "emerg" in dept:
        base = 1
    elif "ortho" in dept or "ped" in dept:
        base = 3
    else:
        base = 4
    if wait_time < 15:
        severity = base
    elif wait_time < 30:
        severity = min(5, base + 1)
    elif wait_time < 60:
        severity = min(5, base + 2)
    else:
        severity = min(5, base + 3)
    return max(1, min(5, severity))

csv_path = find_data_file()
df_raw = pd.read_csv(csv_path)

# Rename columns
df = df_raw.rename(columns={
    "Patient Id": "patient_id",
    "Patient Admission Date": "timestamp",
    "Patient Age": "age",
    "Patient Gender": "gender",
    "Patient Waittime": "wait_time",
    "Department Referral": "department",
    "Patient Admission Flag": "admitted",
    "Patient Satisfaction Score": "satisfaction_score",
    "Patient Race": "race"
})
df["department"] = df["department"].fillna("Self-Referral")
df["satisfaction_score"] = df["satisfaction_score"].fillna(3.0)
df["parsed_timestamp"] = pd.to_datetime(df["timestamp"], format="%d-%m-%Y %H:%M")
df = df.sort_values(by="parsed_timestamp").reset_index(drop=True)

# Generate standard features
df["hour"] = df["parsed_timestamp"].dt.hour
df["day_of_week"] = df["parsed_timestamp"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["gender"] = df["gender"].apply(lambda g: 1 if str(g).lower() == "m" else 0)
df["emergency_severity_level"] = df.apply(lambda r: derive_severity_level(r["wait_time"], r["department"]), axis=1)

print("Generating deterministic synthetic telemetry features...")
fake = Faker()
icu_beds_list, ambulance_list, doctors_list, oxygen_list = [], [], [], []

for idx, row in df.iterrows():
    seed = get_deterministic_seed(row["timestamp"])
    fake.seed_instance(seed)
    icu_beds_list.append(fake.random_int(min=0, max=50))
    ambulance_list.append(fake.random_int(min=0, max=10))
    doctors_list.append(fake.random_int(min=5, max=30))
    oxygen_list.append(round(fake.random.uniform(40.0, 100.0), 2))

df["icu_beds_available"] = icu_beds_list
df["ambulance_requests"] = ambulance_list
df["doctor_availability"] = doctors_list
df["oxygen_utilization"] = oxygen_list
df["admission_target"] = df["admitted"].astype(int)

depts = ["Self-Referral", "Cardiology", "ICU", "Emergency", "Orthopedics", "Pediatrics"]
df["department_encoded"] = df["department"].apply(lambda d: depts.index(d) if d in depts else 0)

print("Data loaded and base pipeline recreated. Shape:", df.shape)

## 3. Advanced Feature Engineering (Targeting >90% Accuracy)

To achieve a very high accuracy rate (such as >90%), we need predictive signals beyond standard fields. We will construct cyclical time coordinates, operational load indicators, and safety ratios.

In [ ]:
# 1. Cyclical time encodings
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7.0)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7.0)

# 2. Department-wise rolling patient counts (representing local load)
df["dept_rolling_count_3h"] = df.groupby("department")["patient_id"].transform(lambda x: x.rolling("3h", on=df["parsed_timestamp"]).count())

# 3. Ratios & clinical interactions
df["age_severity_index"] = df["age"] * (6 - df["emergency_severity_level"])
df["beds_per_doctor"] = df["icu_beds_available"] / (df["doctor_availability"] + 0.1)
df["oxygen_per_icu_bed"] = df["oxygen_utilization"] / (df["icu_beds_available"] + 0.1)
df["wait_time_per_severity"] = df["wait_time"] / (df["emergency_severity_level"] + 0.1)

print("Advanced features generated. Advanced columns preview:")
print(df[["hour_sin", "dept_rolling_count_3h", "age_severity_index", "beds_per_doctor"]].head())

## 4. Selecting Feature Setup

We offer two options below:
1. **Baseline Mode (Compatible):** Matches the existing production app schema exactly.
2. **Advanced Mode (Optimized):** Employs interaction features and rolling metrics to target maximum accuracy. *(Requires updating `app/ml/inference.py` to match)*

In [ ]:
# Toggle feature selector here:
USE_ADVANCED = True

if USE_ADVANCED:
    features_list = [
        "age", "gender", "emergency_severity_level", "hour", "day_of_week",
        "is_weekend", "wait_time", "department_encoded", "icu_beds_available",
        "ambulance_requests", "doctor_availability", "oxygen_utilization",
        "hour_sin", "hour_cos", "day_sin", "day_cos", 
        "dept_rolling_count_3h", "age_severity_index", "beds_per_doctor", "oxygen_per_icu_bed",
        "wait_time_per_severity"
    ]
else:
    features_list = [
        "age", "gender", "emergency_severity_level", "hour", "day_of_week",
        "is_weekend", "wait_time", "department_encoded", "icu_beds_available",
        "ambulance_requests", "doctor_availability", "oxygen_utilization"
    ]

# Ensure department matches integer encoding variable
X = df[features_list].rename(columns={"department_encoded": "department"})
y = df["admission_target"]

# Temporal split
split_idx = int(len(df) * 0.75)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

## 5. Optuna Hyperparameter Optimization with GPU

Using Kaggle's GPU environment, we can run 100+ hyperparameter trials in seconds. We configure XGBoost with `tree_method='hist'` and `device='cuda'` to run on the graphics card.

In [ ]:
def objective(trial):
    params = {
        # GPU Acceleration Configuration
        "tree_method": "hist",
        "device": "cuda",
        
        # Hyperparameter Search Space
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "gamma": trial.suggest_float("gamma", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5, 5.0),
        "eval_metric": "logloss"
    }
    
    # Time Series cross-validation to prevent leakage
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    
    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict(X_val)
        scores.append(accuracy_score(y_val, preds))
        
    return np.mean(scores)

# Create study and optimize
print("Starting Optuna search using GPU...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best Trial Accuracy Score:", study.best_value)
print("Best Trial Parameters:", study.best_params)

## 6. Train Final XGBoost Classifier & Evaluate

Using the optimal hyperparameters discovered by Optuna, we train our final XGBoost classifier on the full training dataset and evaluate on the holdout test set.

In [ ]:
best_params = study.best_params
best_params["tree_method"] = "hist"
best_params["device"] = "cuda"
best_params["eval_metric"] = "logloss"

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Predictions
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("=== Final Evaluation Results ===")
print(f"Accuracy Score: {accuracy:.4f}")
print(f"F1 Score:       {f1:.4f}")
print(f"ROC-AUC Score:  {roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save XGBoost Model
joblib.dump(final_model, "model_xgb.pkl")
print("\nModel successfully serialized and saved to 'model_xgb.pkl'.")

## 7. Prophet forecaster tuning & training

Now we group the dataset to hourly resampled time series and configure Prophet to forecast admissions trends.

In [ ]:
print("Aggregating hourly counts for Prophet forecaster...")
hourly_df = df.resample("H", on="parsed_timestamp").size().reset_index(name="y")
hourly_df = hourly_df.rename(columns={"parsed_timestamp": "ds"})

# Split off last 24 hours as test target
train_forecast_df = hourly_df.iloc[:-24].copy()
test_forecast_df = hourly_df.iloc[-24:].copy()

# Tune Prophet growth and scales
prophet_model = Prophet(
    growth="linear",
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=True,
    changepoint_prior_scale=0.05, # Control trend flexibility
    seasonality_prior_scale=10.0 # Control seasonal flexibility
)

prophet_model.fit(train_forecast_df)

# Predict on test set
forecast = prophet_model.predict(test_forecast_df[["ds"]])

actuals = test_forecast_df["y"].values
preds = forecast["yhat"].values

mae = np.mean(np.abs(actuals - preds))
print(f"Prophet Test set MAE: {mae:.2f} patients/hour")

# Save Prophet Model
joblib.dump(prophet_model, "model_prophet.pkl")
print("Prophet model successfully serialized and saved to 'model_prophet.pkl'.")

## 8. Exporting the Models for MediFlow API Deployment

After training completed on Kaggle:
1. Go to the **Output** tab in your Kaggle notebook.
2. Download `model_xgb.pkl` and `model_prophet.pkl`.
3. Move/paste them into your local project workspace under `app/ml/models/`.
4. Start/Restart the Uvicorn server to load your newly tuned high-accuracy models!